In [ ]:
# ==============================================
# 셀 1: 패키지 설치 및 임포트 테스트
# ==============================================

In [1]:
import os   
import json
import numpy as np
from dotenv import load_dotenv
from typing import Dict, List, TypedDict, Literal    
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# ==============================================
# 셀 2: 데이터 구조 정의
# ==============================================

class ConversationState(TypedDict):
    """
    대화 상태를 관리하는 TypedDict 클래스
    LangGraph의 각 노드 간에 공유되는 상태 정보를 저장

    messages: List[Dict[str, str]]          -> 전체 대화 히스토리
    current_message: str                    ->  현재 처리 중인 사용자 메시지
    selected_task: str                      ->  현재 사용자 메세지에 가장 부합하는 평가 영역
    task_message_relevance: float           ->  메시지와 평가 영역 간 맥락 관련성 
    generated_questions: List[str]          ->  AI가 생성한 평가 질문 후보들
    selected_question: str                  ->  최종 선택된 평가 질문
    question_sample_simliarity: List[float] ->  참조 질문과 선택된 평가 질문 간 유사도 
    question_message_relevance: float       ->  질문과 평가영역 맥락 관련성 
    user_answer: float                      ->  사용자 답변 (채점에 사용)
    conversation_mode: Literal["assessment", "casual"] -> 현재 대화 모드 (검사 vs 일상)

    """
    messages: List[Dict[str, str]]          
    current_message: str                    
    selected_task: str                      
    task_message_relevance: float           
    generated_questions: List[str]          
    selected_question: str                 
    question_sample_simliarity: List[float] 
    question_message_relevance: float      
    user_answer_score: float                     
    conversation_mode: Literal["assessment", "casual"] 


@dataclass
class ChatbotConfig:
    """
    챗봇 설정을 관리하는 데이터클래스
    """
    openai_api_key: str                     # OpenAI API 키 (필수)
    assessment_threshold: float = 0.6       # 평가 vs 일상 대화 진입 기준
    fallback_threshold: float = 0.7         # 질문 재생성 vs 출력 진입 기준
    model_name: str = "gpt-4o-mini"         # 사용할 OpenAI 모델명

In [3]:
# ==============================================
# 셀 3: 평가 영역 및 예시 질문 정의
# ==============================================

ASSESSMENT_TASKS = {

    "registration_recall": {
        "description": 
        """기억 등록은 즉각적인 기억력을 평가하고, 회상은 기억을 유지하는 능력을 평가하는 항목입니다.
          messages 최근 5개 turn 내에서 동일 선상에서 비교될 수 있는 단어/고유명사가 3개 이상 나오면 본 평가내역을 활용할 수 있습니다.""",
        "example_questions": [
            "아까 말씀하신 과일 중 사과, 배, 포도를 어릴 때 가장 좋아했던 순서대로 말씀해주세요.",
            "아까 말씀하신 자녀 중 영희, 철수, 길동이를 살가운 순서대로 말씀해주시겠어요?",
            "아까 말씀하신 공책, 필통, 샤프를 어릴 적 갖고 싶었던 순서대로 말씀해주세요.",
            "콩, 생선, 고추들을 어릴 적 싫어했던 순서대로 말씀해주세요.",
            "콩, 생선, 고추들을 요즘 좋아하시는 순서대로 말씀해주세요."
        ]
    },
    # object detection이 추후 구현되어야 합니다.
    "Naming": {
        "description": 
        """표시된 사물의 이름을 기억해내는 능력을 평가합니다. 
        사진데이터에서 위치관계가 명확한 사물이 있을 경우 본 평가 항목을 사용하기 적당합니다.""",
        "example_questions": [
            "사진 속 어린아이가 들고있는 물체를 뭐라고 불러요?",
            "손가락에 끼고 있는 것의 이름은 뭔가요?",
            "친구가 가지고 놀고 있는 물건의 이름은 뭐에요?",
            "사진 속 할머니 옆에 있는 꽃의 이름은 뭔가요?",
            "아이가 안고 있는 동물의 이름은 뭐에요?",
            "케이크 밑에 있는 가구 이름은 뭔가요?"
            ]
    },
    # 온도 조정 필요 (응용 최소화)
    "time_orientation": {
        "description": 
        """현재 자신이 놓여있는 시간, 날짜, 계절 등의 상황을 올바르게 인식하는 능력을 평가합니다.
        시간 관련 humanmassage가 본 평가항목에 대한 트리거가 됩니다.
        example_questions의 응용을 최소화하여 질문을 생성하세요.
        """,
        "example_questions": [
            "올해는 몇년도인가요?"
        ]
    }
}

In [28]:
# ==============================================
# 셀 4: OpenAI API 연결 테스트 (Just Test)
# ==============================================

# API 키 설정
load_dotenv()
API_KEY = os.getenv("GPT_API_KEY")

# API 연결 테스트
print("OpenAI API 연결 테스트 중...")

try:
    # 테스트용 LLM 객체 생성
    test_llm = ChatOpenAI(
        model="gpt-4o-mini",
        openai_api_key=API_KEY,
        temperature=0.3
    )
    
    # 간단한 테스트 메시지
    test_prompt = ChatPromptTemplate.from_template("안녕하세요라고 한국어로 답변해주세요.")
    test_response = test_llm.invoke(test_prompt.format_messages())
    
    print("✅ OpenAI API 연결 성공!")
    print(f"테스트 응답: {test_response.content}")
    
except Exception as e:
    print(f"❌ OpenAI API 연결 실패: {e}")
    print("\n해결 방법:")
    print("1. API_KEY 변수에 실제 OpenAI API 키를 입력하세요")
    print("2. 인터넷 연결을 확인하세요")
    print("3. API 키가 유효하고 잔액이 있는지 확인하세요")

OpenAI API 연결 테스트 중...
✅ OpenAI API 연결 성공!
테스트 응답: 안녕하세요! 어떻게 도와드릴까요?


In [4]:
# API 키 설정
load_dotenv()
API_KEY = os.getenv("GPT_API_KEY")

# ==============================================
# 셀 5: 챗봇 클래스 정의 (기본 구조)
# ==============================================

class DementiaAssessmentChatbot:
    """
    치매 평가 챗봇 메인 클래스
    """
    
    def __init__(self, config: ChatbotConfig):
        """챗봇 초기화"""
        self.config = config
        self.llm = ChatOpenAI(
            model=config.model_name,
            openai_api_key=config.openai_api_key,
            temperature=0.3
        )   
        
        self.vectorizer = TfidfVectorizer(stop_words='english')
        print(f"✅ 챗봇 초기화 완료 (모델: {config.model_name})")
    
    def test_llm_connection(self):
        """LLM 연결 테스트"""
        try:
            prompt = ChatPromptTemplate.from_template("테스트 메시지입니다. '연결 성공'이라고 답변해주세요.")
            response = self.llm.invoke(prompt.format_messages())
            print(f"✅ LLM 연결 테스트 성공: {response.content}")
            return True
        except Exception as e:
            print(f"❌ LLM 연결 테스트 실패: {e}")
            return False


In [5]:
# ==============================================
# 셀 6: 기본 챗봇 초기화 테스트
# ==============================================

# 설정 객체 생성 (여기 threshold 조정 중요)
config = ChatbotConfig(
    openai_api_key=API_KEY,  
    assessment_threshold=0.8,
    fallback_threshold=0.7
)


# 챗봇 인스턴스 생성 및 테스트
try:
    chatbot = DementiaAssessmentChatbot(config)
    chatbot.test_llm_connection()
    print("✅ 기본 챗봇 설정 완료!")
except Exception as e:
    print(f"❌ 챗봇 초기화 실패: {e}")


✅ 챗봇 초기화 완료 (모델: gpt-4o-mini)
✅ LLM 연결 테스트 성공: 연결 성공!
✅ 기본 챗봇 설정 완료!


In [ ]:
# ==============================================
# 셀 7: 개별 메서드 테스트 - 태스크 유사도 계산
# ==============================================

class DementiaAssessmentChatbot_Step1(DementiaAssessmentChatbot):
    """1단계: 태스크 별 적합도 계산 기능 (human massage가 각 task의 description을 얼마나 만족하는가?)"""
    
    def _calculate_task_similarity(self, state: ConversationState) -> ConversationState:
        # 1. LLM이 각 task와의 적합도 계산
        print(f"🔍 태스크 유사도 계산 중... 메시지: '{state['current_message']}'")
        
        message = state["current_message"]
        task_scores = {}
        
        for task_name, task_info in ASSESSMENT_TASKS.items():
            print(f"  - {task_name} 평가 중...")
            
            prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            평가 영역: {task_name} - {description}
            
            사용자 메시지가 이 평가 영역과 얼마나 관련이 있는지 0-1 사이의 점수로 평가해주세요.
            0: 전혀 관련 없음, 1: 매우 관련 있음
            
            점수만 반환해주세요 (예: 0.8):
            """)
            
            try:
                response = self.llm.invoke(prompt.format_messages(
                    message=message,
                    task_name=task_name,
                    description=task_info["description"]
                ))
                
                score = float(response.content.strip())
                task_scores[task_name] = score
                print(f"    점수: {score:.2f}")
                
            except Exception as e:
                print(f"    오류 발생: {e}")
                task_scores[task_name] = 0.0
        
        # 2. 최적 Task 선택 (노드를 분리-> 재활용 가능): human message와 적합도가 높은 task 선택
        if task_scores:
            selected_task = max(task_scores.items(), key=lambda x: x[1])
            state["selected_task"] = selected_task[0]
            print(f"🎯 선택된 태스크: {selected_task[0]} (점수: {selected_task[1]:.2f})")
        
        return state

# 1단계 테스트
print("=== 1단계 테스트: 태스크 유사도 계산 ===")

chatbot_step1 = DementiaAssessmentChatbot_Step1(config)

# 테스트 메시지들
test_messages = [
    "참외, 오이, 수박을 싸가서 먹었어.",
]

for msg in test_messages:
    print(f"\n--- 테스트 메시지: '{msg}' ---")
    
    test_state = ConversationState(
        messages=[],
        current_message=msg,
        selected_task="",
        task_message_relevance=0.0,
        generated_questions=[],
        selected_question="",
        question_sample_simliarity=[],
        question_message_relevance=0.0,
        user_answer_score=0.0,
        conversation_mode="casual"
    )
    
    result = chatbot_step1._calculate_task_similarity(test_state)
    print(f"결과: {result['selected_task']}\n")


=== 1단계 테스트: 태스크 유사도 계산 ===
✅ 챗봇 초기화 완료 (모델: gpt-4o-mini)

--- 테스트 메시지: '참외, 오이, 수박을 싸가서 먹었어.' ---
🔍 태스크 유사도 계산 중... 메시지: '참외, 오이, 수박을 싸가서 먹었어.'
  - registration_recall 평가 중...
    점수: 0.80
  - Naming 평가 중...
    점수: 1.00
  - time_orientation 평가 중...
    점수: 0.70
🎯 선택된 태스크: Naming (점수: 1.00)
결과: Naming


--- 테스트 메시지: '그 때는 다들 그렇게 지냈어.' ---
🔍 태스크 유사도 계산 중... 메시지: '그 때는 다들 그렇게 지냈어.'
  - registration_recall 평가 중...
    점수: 0.50
  - Naming 평가 중...
    점수: 0.00
  - time_orientation 평가 중...
    점수: 0.70
🎯 선택된 태스크: time_orientation (점수: 0.70)
결과: time_orientation



In [ ]:
# ==============================================
# 셀 8: 최적 Task가 human massage와 맥락적 Threshold를 달성하는지 체크 
# ==============================================

class DementiaAssessmentChatbot_Step2(DementiaAssessmentChatbot_Step1):
    """2단계: 맥락 관련성 확인 기능 추가 (description의 설명 키워드를 human massage가 만족하는지, 위 적합 task가 "정말로" threshold를 넘어 검사 대화를 진행해도 되는지 판별)"""

    def _check_context_relevance(self, state: ConversationState) -> ConversationState:
        """맥락 일치도 확인"""
        print(f"🔍 맥락 관련성 확인 중...")
        
        message = state["current_message"]
        selected_task = state["selected_task"]
        
        if selected_task not in ASSESSMENT_TASKS:
            print(f"알 수 없는 태스크: {selected_task}")
            state["task_message_relevance"] = 0.0
            return state
            
        task_info = ASSESSMENT_TASKS[selected_task]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        평가 영역: {task_name}
        영역 설명: {description}
        
        사용자 메시지가 이 평가 영역의 설명에서 제시한 조건/키워드를 만족하는지 0-1 사이의 점수로 평가해주세요.
        평가 기준:
        - 메시지가 영역 설명에서 언급된 조건들을 포함하고 있는가?
        - 해당 인지 기능 평가를 진행하기에 적절한 상황인가?
        
        점수만 반환해주세요 (예: 0.7):
        """)
        
        try:
            response = self.llm.invoke(prompt.format_messages(
                message=message,
                task_name=selected_task,
                description=task_info["description"]
            ))
            
            relevance_score = float(response.content.strip())
            state["task_message_relevance"] = relevance_score
            print(f"맥락 관련성 점수: {relevance_score:.2f}")
            
        except Exception as e:
            print(f"맥락 관련성 계산 실패: {e}")
            state["task_message_relevance"] = 0.0
            
        return state

# 2단계 테스트
print("=== 2단계 테스트: 맥락 관련성 확인 ===")

chatbot_step2 = DementiaAssessmentChatbot_Step2(config)

for msg in test_messages:
    print(f"\n--- 테스트 메시지: '{msg}' ---")
    
    test_state = ConversationState(
        messages=[],
        current_message=msg,
        selected_task="",
        task_message_relevance=0.0,
        generated_questions=[],
        selected_question="",
        question_sample_simliarity=[],
        question_message_relevance=0.0,
        user_answer_score=0.0,
        conversation_mode="casual"
    )
    
    # 1단계 실행
    result = chatbot_step2._calculate_task_similarity(test_state)
    # 2단계 실행
    result = chatbot_step2._check_context_relevance(result)
    
    print(f"선택된 태스크: {result['selected_task']}")
    print(f"맥락 관련성: {result['task_message_relevance']:.2f}")
    
    # 임계값 확인
    threshold = config.assessment_threshold
    if result['task_message_relevance'] >= threshold:
        print(f"✅ 평가 모드 진입 (임계값 {threshold} 이상)")
    else:
        print(f"💬 일상 대화 모드 (임계값 {threshold} 미만)")


=== 2단계 테스트: 맥락 관련성 확인 ===
✅ 챗봇 초기화 완료 (모델: gpt-4o-mini)

--- 테스트 메시지: '참외, 오이, 수박을 싸가서 먹었어.' ---
🔍 태스크 유사도 계산 중... 메시지: '참외, 오이, 수박을 싸가서 먹었어.'
  - registration_recall 평가 중...
    점수: 1.00
  - Naming 평가 중...
    점수: 1.00
  - time_orientation 평가 중...
    점수: 0.70
🎯 선택된 태스크: registration_recall (점수: 1.00)
🔍 맥락 관련성 확인 중...
맥락 관련성 점수: 0.80
선택된 태스크: registration_recall
맥락 관련성: 0.80
✅ 평가 모드 진입 (임계값 0.8 이상)

--- 테스트 메시지: '그 때는 다들 그렇게 지냈어.' ---
🔍 태스크 유사도 계산 중... 메시지: '그 때는 다들 그렇게 지냈어.'
  - registration_recall 평가 중...
    점수: 0.50
  - Naming 평가 중...
    점수: 0.00
  - time_orientation 평가 중...
    점수: 0.60
🎯 선택된 태스크: time_orientation (점수: 0.60)
🔍 맥락 관련성 확인 중...
맥락 관련성 점수: 0.50
선택된 태스크: time_orientation
맥락 관련성: 0.50
💬 일상 대화 모드 (임계값 0.8 미만)


In [ ]:
# ==============================================
# 셀 9: 질문 생성 및 선택 테스트 (assessment 대화 실행 시 노드)
# ==============================================

class DementiaAssessmentChatbot_Step3(DementiaAssessmentChatbot_Step2):
    """3단계: 질문 생성 및 선택 기능 추가"""
    
    def _generate_assessment_questions(self, state: ConversationState) -> ConversationState:
        # 1. "AI 질문 생성 (5개 정도)
        print(f"🤖 평가 질문 생성 중...")
        
        message = state["current_message"]
        selected_task = state["selected_task"]
        task_info = ASSESSMENT_TASKS[selected_task]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        평가 영역: {task_name}
        예시 질문들: {examples}
        
        위 사용자 메시지의 맥락을 고려하여, {task_name} 평가를 위한 자연스러운 질문 5개를 생성해주세요.
        예시 질문들을 참고하되, 사용자의 메시지와 자연스럽게 이어지도록 만들어주세요.
        
        각 질문을 새 줄로 구분하여 번호 없이 나열해주세요:
        """)
        
        try:
            response = self.llm.invoke(prompt.format_messages(
                message=message,
                task_name=selected_task,
                examples="\n".join(task_info["example_questions"])
            ))
            
            generated_questions = [q.strip() for q in response.content.split('\n') if q.strip()]
            state["generated_questions"] = generated_questions
            
            print(f"📝 생성된 질문 {len(generated_questions)}개:")
            for i, q in enumerate(generated_questions, 1):
                print(f"  {i}. {q}")
                
        except Exception as e:
            print(f"❌ 질문 생성 실패: {e}")
            state["generated_questions"] = []
        
        return state
    
    # 2. 각 질문과 예시 문항 간 유사성 비교 및 가장 유사도 높은 질문 선택
    def _select_best_question(self, state: ConversationState) -> ConversationState:
        print(f"🎯 최적 질문 선택 중...")
        
        generated_questions = state["generated_questions"]
        selected_task = state["selected_task"]
        
        if not generated_questions:
            print("❌ 생성된 질문이 없습니다.")
            state["selected_question"] = ""
            state["question_sample_simliarity"] = []
            return state
            
        example_questions = ASSESSMENT_TASKS[selected_task]["example_questions"]
        
        # 모든 질문을 벡터화
        all_questions = generated_questions + example_questions
        
        try:
            tfidf_matrix = self.vectorizer.fit_transform(all_questions)
            
            # 생성된 질문들과 예시 질문들 간의 유사도 계산
            generated_vectors = tfidf_matrix[:len(generated_questions)]
            example_vectors = tfidf_matrix[len(generated_questions):]
            
            similarity_matrix = cosine_similarity(generated_vectors, example_vectors)
            
            # 각 생성된 질문의 최대 유사도 점수 계산
            max_similarities = np.max(similarity_matrix, axis=1)
            
            # 가장 높은 유사도를 가진 질문 선택
            best_question_idx = np.argmax(max_similarities)
            selected_question = generated_questions[best_question_idx]
            
            state["selected_question"] = selected_question
            state["question_sample_simliarity"] = max_similarities.tolist()
            
            print(f"📊 질문별 유사도 점수:")
            for i, (q, score) in enumerate(zip(generated_questions, max_similarities)):
                marker = "👑" if i == best_question_idx else "  "
                print(f"  {marker} {score:.3f}: {q}")
                
        except Exception as e:
            print(f"❌ 질문 선택 실패: {e}")
            state["selected_question"] = generated_questions[0] if generated_questions else ""
            state["question_sample_simliarity"] = [0.0]
            
        return state
    
    def _validate_question_context(self, state: ConversationState) -> ConversationState:
        """
        4단계: 선택된 질문이 사용자 메시지와 맥락적으로 적합한지 검증
        """
        print(f"질문-메시지 맥락 적합성 검증 중...")
        
        current_message = state["current_message"]
        selected_question = state["selected_question"]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{current_message}"
        생성된 질문: "{selected_question}"
        
        이 질문이 사용자 메시지와 자연스럽게 이어지는 대화인지 0-1 사이의 점수로 평가해주세요.
        평가 기준:
        - 대화의 자연스러운 흐름
        - 맥락의 연결성  
        - 갑작스럽지 않은 전환
        - 사용자가 답변할 수 있는 적절한 질문인지
        
        점수만 반환해주세요 (예: 0.8):
        """)
        
        try:
            response = self.llm.invoke(prompt.format_messages(
                current_message=current_message,
                selected_question=selected_question
            ))
            
            validity_score = float(response.content.strip())
            state["question_message_relevance"] = validity_score
            print(f"질문-메시지 맥락 점수: {validity_score:.2f}")
            
        except Exception as e:
            print(f"맥락 검증 실패: {e}")
            state["question_message_relevance"] = 0.0
            
        return state
    
    # 다시 질문 생성하거나 일상 대화로 전환
    def _route_question_output(self, state: ConversationState) -> Literal["output_question", "fallback_generate"]:
        """타당도에 따른 질문 출력 여부 결정"""
        if state["question_message_relevance"] >= self.config.fallback_threshold:
            return "output_question"
        else:
            return "fallback_generate"  


# 3단계 테스트
print("=== 3단계 테스트: 질문 생성 및 선택 ===")

chatbot_step3 = DementiaAssessmentChatbot_Step3(config)

# 평가 모드로 진입할 만한 메시지로 테스트
evaluation_message = "참외, 오이, 수박을 싸가서 먹었어"

print(f"\n--- 평가 질문 생성 테스트: '{evaluation_message}' ---")

test_state = ConversationState(
    messages=[],
    current_message=evaluation_message,
    selected_task="",
    task_message_relevance=0.0,
    generated_questions=[],
    selected_question="",
    question_sample_simliarity=[],
    question_message_relevance=0.0,
    user_answer_score=0.0,
    conversation_mode="casual"
)

# 1-2단계 실행
result = chatbot_step3._calculate_task_similarity(test_state)
result = chatbot_step3._check_context_relevance(result)

# 평가 모드 진입 조건 확인
if result['task_message_relevance'] >= config.assessment_threshold:
    print(f"✅ 평가 모드 진입! 질문 생성을 시작합니다.")
    
    # 3단계 실행
    result = chatbot_step3._generate_assessment_questions(result)
    result = chatbot_step3._select_best_question(result)
    
    print(f"\n🎯 최종 선택된 질문: {result['selected_question']}")
else:
    print(f"💬 일상 대화 모드 (맥락 점수: {result['task_message_relevance']:.2f})")

=== 3단계 테스트: 질문 생성 및 선택 ===
✅ 챗봇 초기화 완료 (모델: gpt-4o-mini)

--- 평가 질문 생성 테스트: '참외, 오이, 수박을 싸가서 먹었어' ---
🔍 태스크 유사도 계산 중... 메시지: '참외, 오이, 수박을 싸가서 먹었어'
  - registration_recall 평가 중...
    점수: 1.00
  - Naming 평가 중...
    점수: 1.00
  - time_orientation 평가 중...
    점수: 0.70
🎯 선택된 태스크: registration_recall (점수: 1.00)
🔍 맥락 관련성 확인 중...
맥락 관련성 점수: 1.00
✅ 평가 모드 진입! 질문 생성을 시작합니다.
🤖 평가 질문 생성 중...
📝 생성된 질문 5개:
  1. 참외, 오이, 수박 중 어릴 적 가장 좋아했던 순서대로 말씀해주시겠어요?
  2. 아까 말씀하신 과일과 채소 중에서 요즘 가장 자주 먹는 순서대로 알려주실 수 있나요?
  3. 참외, 오이, 수박을 먹었던 기억 중 가장 특별한 순간을 말씀해주실 수 있나요?
  4. 어릴 적에 참외, 오이, 수박 중에서 가장 싫어했던 것을 순서대로 말씀해주실 수 있나요?
  5. 참외, 오이, 수박을 먹으면서 떠오르는 추억이 있다면 이야기해주실 수 있나요?
🎯 최적 질문 선택 중...
📊 질문별 유사도 점수:
  👑 0.336: 참외, 오이, 수박 중 어릴 적 가장 좋아했던 순서대로 말씀해주시겠어요?
     0.203: 아까 말씀하신 과일과 채소 중에서 요즘 가장 자주 먹는 순서대로 알려주실 수 있나요?
     0.054: 참외, 오이, 수박을 먹었던 기억 중 가장 특별한 순간을 말씀해주실 수 있나요?
     0.269: 어릴 적에 참외, 오이, 수박 중에서 가장 싫어했던 것을 순서대로 말씀해주실 수 있나요?
     0.000: 참외, 오이, 수박을 먹으면서 떠오르는 추억이 있다면 이야기해주실 수 있나요?

🎯 최종 선택된 질문: 참외, 오이, 수박 중 어릴 적 가

### 8번 threshold 기준 변경 -> 처리 속도 향상

In [ ]:
""" 
task가 진짜 human massage가 만족하는지 description의 설명만으로 예상 유도
-> human message를 example과 비교해 처리 시간 긺
-> 상위 3개 example이 표준이 아니기 때문에 예측 정확도 낮음
-> 표준 description과만 일관되게 비교하도록 변경

"""



# ==============================================
# 셀 1: 패키지 설치 및 임포트 테스트
# ==============================================


# 먼저 이 셀을 실행해서 모든 패키지가 정상적으로 임포트되는지 확인
print("패키지 임포트 테스트 시작...")

try:
    import os
    print("✅ os 임포트 성공")
    
    from typing import Dict, List, TypedDict, Literal
    print("✅ typing 임포트 성공")
    
    from dataclasses import dataclass
    print("✅ dataclasses 임포트 성공")
    
    from langchain_openai import ChatOpenAI
    print("✅ langchain_openai 임포트 성공")
    
    from langchain_core.messages import HumanMessage, AIMessage
    print("✅ langchain_core.messages 임포트 성공")
    
    from langchain_core.prompts import ChatPromptTemplate
    print("✅ langchain_core.prompts 임포트 성공")
    
    from langgraph.graph import StateGraph, END
    print("✅ langgraph.graph 임포트 성공")
    
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    print("✅ sklearn 임포트 성공")
    
    import numpy as np
    print("✅ numpy 임포트 성공")
    
    import json
    print("✅ json 임포트 성공")
    
    print("\n🎉 모든 패키지 임포트 성공!")
    
except ImportError as e:
    print(f"❌ 임포트 실패: {e}")
    print("\n해결 방법:")
    print("1. 터미널에서 'uv sync' 실행")
    print("2. Jupyter 커널을 uv 가상환경으로 변경")
    print("3. 패키지 설치: uv add langchain-openai scikit-learn numpy")

# ==============================================
# 셀 2: 데이터 구조 정의
# ==============================================

class ConversationState(TypedDict):
    """
    대화 상태를 관리하는 TypedDict 클래스
    LangGraph의 각 노드 간에 공유되는 상태 정보를 저장
    """
    messages: List[Dict[str, str]]          # 전체 대화 히스토리
    current_message: str                    # 현재 처리 중인 사용자 메시지
    selected_task: str                      # LLM이 선택한 가장 적합한 평가 영역
    context_relevance_score: float         # 메시지와 평가 영역 간 맥락 관련성 점수 (0-1)
    generated_questions: List[str]          # AI가 생성한 평가 질문 후보들
    selected_question: str                  # 최종 선택된 평가 질문
    similarity_scores: List[float]          # 생성된 질문들과 예시 질문들 간의 유사도 점수
    context_validity_score: float          # 선택된 질문이 대화 맥락에 적합한지 타당성 점수 (0-1)
    assessment_score: float                 # 사용자 답변에 대한 평가 점수 (0-10)
    conversation_mode: Literal["assessment", "casual"]  # 현재 대화 모드


@dataclass
class ChatbotConfig:
    """
    챗봇 설정을 관리하는 데이터클래스
    """
    openai_api_key: str                     # OpenAI API 키 (필수)
    context_threshold: float = 0.6          # 맥락 관련성 임계값
    validity_threshold: float = 0.7         # 문맥 타당성 임계값
    model_name: str = "gpt-4o-mini"         # 사용할 OpenAI 모델명

print("✅ 데이터 구조 정의 완료")

# ==============================================
# 셀 3: 평가 영역 및 예시 질문 정의
# ==============================================

ASSESSMENT_TASKS = {
    "기억력_평가": {
        "description": "최근 기억, 장기 기억, 단어 기억 등을 평가",
        "example_questions": [
            "어제 저녁에 무엇을 드셨나요?",
            "오늘 아침에 누구와 함께 계셨나요?",
            "지금 제가 말씀드린 세 가지 단어를 기억하고 계신가요?",
            "어릴 때 살던 동네 이름을 기억하시나요?",
            "결혼식은 언제 하셨나요?"
        ]
    },
    "시간_지남력": {
        "description": "현재 시간, 날짜, 계절 등에 대한 인지 능력 평가",
        "example_questions": [
            "오늘이 며칠인지 아시나요?",
            "지금이 몇 월인지 말씀해 주세요",
            "현재 계절이 무엇인지 아시나요?",
            "오늘이 무슨 요일인지 기억하시나요?",
            "지금 몇 시쯤 되는 것 같나요?"
        ]
    },
    "언어_능력": {
        "description": "언어 이해, 표현, 명명 능력 평가",
        "example_questions": [
            "이 물건의 이름이 무엇인가요?",
            "제가 하는 말을 따라해 주세요",
            "반대말을 말씀해 주세요. '크다'의 반대는?",
            "이 그림에서 무엇을 보시나요?",
            "문장을 완성해 주세요: '아침에 일어나서 가장 먼저 하는 일은...'"
        ]
    }
}

print("✅ 평가 영역 정의 완료")
print(f"등록된 평가 영역: {list(ASSESSMENT_TASKS.keys())}")

# ==============================================
# 셀 4: OpenAI API 연결 테스트
# ==============================================

# API 키 설정 (실제 키로 변경하세요)
API_KEY = "your-openai-api-key-here"  # 실제 API 키로 변경!

# API 연결 테스트
print("OpenAI API 연결 테스트 중...")

try:
    # 테스트용 LLM 객체 생성
    test_llm = ChatOpenAI(
        model="gpt-4o-mini",
        openai_api_key=API_KEY,
        temperature=0.3
    )
    
    # 간단한 테스트 메시지
    test_prompt = ChatPromptTemplate.from_template("안녕하세요라고 한국어로 답변해주세요.")
    test_response = test_llm.invoke(test_prompt.format_messages())
    
    print("✅ OpenAI API 연결 성공!")
    print(f"테스트 응답: {test_response.content}")
    
except Exception as e:
    print(f"❌ OpenAI API 연결 실패: {e}")
    print("\n해결 방법:")
    print("1. API_KEY 변수에 실제 OpenAI API 키를 입력하세요")
    print("2. 인터넷 연결을 확인하세요")
    print("3. API 키가 유효하고 잔액이 있는지 확인하세요")

# ==============================================
# 셀 5: 챗봇 클래스 정의 (기본 구조)
# ==============================================

class DementiaAssessmentChatbot:
    """
    치매 평가 챗봇 메인 클래스
    """
    
    def __init__(self, config: ChatbotConfig):
        """챗봇 초기화"""
        self.config = config
        self.llm = ChatOpenAI(
            model=config.model_name,
            openai_api_key=config.openai_api_key,
            temperature=0.3
        )   
        self.vectorizer = TfidfVectorizer(stop_words='english')
        print(f"✅ 챗봇 초기화 완료 (모델: {config.model_name})")
    
    def test_llm_connection(self):
        """LLM 연결 테스트"""
        try:
            prompt = ChatPromptTemplate.from_template("테스트 메시지입니다. '연결 성공'이라고 답변해주세요.")
            response = self.llm.invoke(prompt.format_messages())
            print(f"✅ LLM 연결 테스트 성공: {response.content}")
            return True
        except Exception as e:
            print(f"❌ LLM 연결 테스트 실패: {e}")
            return False

print("✅ 챗봇 클래스 기본 구조 정의 완료")

# ==============================================
# 셀 6: 기본 챗봇 테스트
# ==============================================

# 설정 객체 생성
config = ChatbotConfig(
    openai_api_key=API_KEY,  # 위에서 설정한 API 키 사용
    context_threshold=0.6,
    validity_threshold=0.7
)

# 챗봇 인스턴스 생성 및 테스트
try:
    chatbot = DementiaAssessmentChatbot(config)
    chatbot.test_llm_connection()
    print("✅ 기본 챗봇 설정 완료!")
except Exception as e:
    print(f"❌ 챗봇 초기화 실패: {e}")

# ==============================================
# 셀 7: 개별 메서드 테스트 - 태스크 유사도 계산
# ==============================================

class DementiaAssessmentChatbot_Step1(DementiaAssessmentChatbot):
    """1단계: 태스크 유사도 계산 기능 추가"""
    
    def _calculate_task_similarity(self, state: ConversationState) -> ConversationState:
        """1. LLM이 각 task와의 부합도 계산"""
        print(f"🔍 태스크 유사도 계산 중... 메시지: '{state['current_message']}'")
        
        message = state["current_message"]
        task_scores = {}
        
        for task_name, task_info in ASSESSMENT_TASKS.items():
            print(f"  - {task_name} 평가 중...")
            
            prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            평가 영역: {task_name} - {description}
            
            사용자 메시지가 이 평가 영역과 얼마나 관련이 있는지 0-1 사이의 점수로 평가해주세요.
            0: 전혀 관련 없음, 1: 매우 관련 있음
            
            점수만 반환해주세요 (예: 0.8):
            """)
            
            try:
                response = self.llm.invoke(prompt.format_messages(
                    message=message,
                    task_name=task_name,
                    description=task_info["description"]
                ))
                
                score = float(response.content.strip())
                task_scores[task_name] = score
                print(f"    점수: {score:.2f}")
                
            except Exception as e:
                print(f"    오류 발생: {e}")
                task_scores[task_name] = 0.0
        
        # 가장 높은 점수의 태스크 선택
        if task_scores:
            selected_task = max(task_scores.items(), key=lambda x: x[1])
            state["selected_task"] = selected_task[0]
            print(f"🎯 선택된 태스크: {selected_task[0]} (점수: {selected_task[1]:.2f})")
        
        return state

# 1단계 테스트
print("=== 1단계 테스트: 태스크 유사도 계산 ===")

chatbot_step1 = DementiaAssessmentChatbot_Step1(config)

# 테스트 메시지들
test_messages = [
    "어제 뭘 먹었는지 기억이 안 나요",
    "오늘이 몇 월인지 모르겠어요",
    "단어가 잘 생각이 안 나요"
]

for msg in test_messages:
    print(f"\n--- 테스트 메시지: '{msg}' ---")
    
    test_state = ConversationState(
        messages=[],
        current_message=msg,
        selected_task="",
        context_relevance_score=0.0,
        generated_questions=[],
        selected_question="",
        similarity_scores=[],
        context_validity_score=0.0,
        assessment_score=0.0,
        conversation_mode="casual"
    )
    
    result = chatbot_step1._calculate_task_similarity(test_state)
    print(f"결과: {result['selected_task']}\n")

# ==============================================
# 셀 8: 맥락 관련성 확인 테스트
# ==============================================

class DementiaAssessmentChatbot_Step2(DementiaAssessmentChatbot_Step1):
    """2단계: 맥락 관련성 확인 기능 추가"""
    
    def _check_context_relevance(self, state: ConversationState) -> ConversationState:
        """맥락 일치도 확인"""
        print(f"🔍 맥락 관련성 확인 중...")
        
        message = state["current_message"]
        selected_task = state["selected_task"]
        
        if selected_task not in ASSESSMENT_TASKS:
            print(f"❌ 알 수 없는 태스크: {selected_task}")
            state["context_relevance_score"] = 0.0
            return state
            
        task_info = ASSESSMENT_TASKS[selected_task]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        평가 영역: {task_name} - {description}
        예시 질문들: {examples}
        
        사용자 메시지가 이 평가 영역의 맥락에 얼마나 적합한지 0-1 사이의 점수로 평가해주세요.
        평가 기준:
        - 메시지가 해당 인지 영역과 관련된 내용인가?
        - 자연스러운 평가로 이어질 수 있는 대화인가?
        
        점수만 반환해주세요 (예: 0.7):
        """)
        
        try:
            response = self.llm.invoke(prompt.format_messages(
                message=message,
                task_name=selected_task,
                description=task_info["description"],
                examples=", ".join(task_info["example_questions"][:3])
            ))
            
            relevance_score = float(response.content.strip())
            state["context_relevance_score"] = relevance_score
            print(f"📊 맥락 관련성 점수: {relevance_score:.2f}")
            
        except Exception as e:
            print(f"❌ 맥락 관련성 계산 실패: {e}")
            state["context_relevance_score"] = 0.0
            
        return state

# 2단계 테스트
print("=== 2단계 테스트: 맥락 관련성 확인 ===")

chatbot_step2 = DementiaAssessmentChatbot_Step2(config)

for msg in test_messages:
    print(f"\n--- 테스트 메시지: '{msg}' ---")
    
    test_state = ConversationState(
        messages=[],
        current_message=msg,
        selected_task="",
        context_relevance_score=0.0,
        generated_questions=[],
        selected_question="",
        similarity_scores=[],
        context_validity_score=0.0,
        assessment_score=0.0,
        conversation_mode="casual"
    )
    
    # 1단계 실행
    result = chatbot_step2._calculate_task_similarity(test_state)
    # 2단계 실행
    result = chatbot_step2._check_context_relevance(result)
    
    print(f"선택된 태스크: {result['selected_task']}")
    print(f"맥락 관련성: {result['context_relevance_score']:.2f}")
    
    # 임계값 확인
    threshold = config.context_threshold
    if result['context_relevance_score'] >= threshold:
        print(f"✅ 평가 모드 진입 (임계값 {threshold} 이상)")
    else:
        print(f"💬 일상 대화 모드 (임계값 {threshold} 미만)")

# ==============================================
# 셀 9: 질문 생성 및 선택 테스트
# ==============================================

class DementiaAssessmentChatbot_Step3(DementiaAssessmentChatbot_Step2):
    """3단계: 질문 생성 및 선택 기능 추가"""
    
    def _generate_assessment_questions(self, state: ConversationState) -> ConversationState:
        """AI 질문 생성 (5개 정도)"""
        print(f"🤖 평가 질문 생성 중...")
        
        message = state["current_message"]
        selected_task = state["selected_task"]
        task_info = ASSESSMENT_TASKS[selected_task]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        평가 영역: {task_name}
        예시 질문들: {examples}
        
        위 사용자 메시지의 맥락을 고려하여, {task_name} 평가를 위한 자연스러운 질문 5개를 생성해주세요.
        예시 질문들을 참고하되, 사용자의 메시지와 자연스럽게 이어지도록 만들어주세요.
        
        각 질문을 새 줄로 구분하여 번호 없이 나열해주세요:
        """)
        
        try:
            response = self.llm.invoke(prompt.format_messages(
                message=message,
                task_name=selected_task,
                examples="\n".join(task_info["example_questions"])
            ))
            
            generated_questions = [q.strip() for q in response.content.split('\n') if q.strip()]
            state["generated_questions"] = generated_questions
            
            print(f"📝 생성된 질문 {len(generated_questions)}개:")
            for i, q in enumerate(generated_questions, 1):
                print(f"  {i}. {q}")
                
        except Exception as e:
            print(f"❌ 질문 생성 실패: {e}")
            state["generated_questions"] = []
        
        return state
    
    def _select_best_question(self, state: ConversationState) -> ConversationState:
        """각 질문과 예시 문항 간 유사성 비교 및 가장 유사도 높은 질문 선택"""
        print(f"🎯 최적 질문 선택 중...")
        
        generated_questions = state["generated_questions"]
        selected_task = state["selected_task"]
        
        if not generated_questions:
            print("❌ 생성된 질문이 없습니다.")
            state["selected_question"] = ""
            state["similarity_scores"] = []
            return state
            
        example_questions = ASSESSMENT_TASKS[selected_task]["example_questions"]
        
        # 모든 질문을 벡터화
        all_questions = generated_questions + example_questions
        
        try:
            tfidf_matrix = self.vectorizer.fit_transform(all_questions)
            
            # 생성된 질문들과 예시 질문들 간의 유사도 계산
            generated_vectors = tfidf_matrix[:len(generated_questions)]
            example_vectors = tfidf_matrix[len(generated_questions):]
            
            similarity_matrix = cosine_similarity(generated_vectors, example_vectors)
            
            # 각 생성된 질문의 최대 유사도 점수 계산
            max_similarities = np.max(similarity_matrix, axis=1)
            
            # 가장 높은 유사도를 가진 질문 선택
            best_question_idx = np.argmax(max_similarities)
            selected_question = generated_questions[best_question_idx]
            
            state["selected_question"] = selected_question
            state["similarity_scores"] = max_similarities.tolist()
            
            print(f"📊 질문별 유사도 점수:")
            for i, (q, score) in enumerate(zip(generated_questions, max_similarities)):
                marker = "👑" if i == best_question_idx else "  "
                print(f"  {marker} {score:.3f}: {q}")
                
        except Exception as e:
            print(f"❌ 질문 선택 실패: {e}")
            state["selected_question"] = generated_questions[0] if generated_questions else ""
            state["similarity_scores"] = [0.0]
            
        return state

# 3단계 테스트
print("=== 3단계 테스트: 질문 생성 및 선택 ===")

chatbot_step3 = DementiaAssessmentChatbot_Step3(config)

# 평가 모드로 진입할 만한 메시지로 테스트
evaluation_message = "어제 뭘 먹었는지 기억이 안 나요"

print(f"\n--- 평가 질문 생성 테스트: '{evaluation_message}' ---")

test_state = ConversationState(
    messages=[],
    current_message=evaluation_message,
    selected_task="",
    context_relevance_score=0.0,
    generated_questions=[],
    selected_question="",
    similarity_scores=[],
    context_validity_score=0.0,
    assessment_score=0.0,
    conversation_mode="casual"
)

# 1-2단계 실행
result = chatbot_step3._calculate_task_similarity(test_state)
result = chatbot_step3._check_context_relevance(result)

# 평가 모드 진입 조건 확인
if result['context_relevance_score'] >= config.context_threshold:
    print(f"✅ 평가 모드 진입! 질문 생성을 시작합니다.")
    
    # 3단계 실행
    result = chatbot_step3._generate_assessment_questions(result)
    result = chatbot_step3._select_best_question(result)
    
    print(f"\n🎯 최종 선택된 질문: {result['selected_question']}")
else:
    print(f"💬 일상 대화 모드 (맥락 점수: {result['context_relevance_score']:.2f})")

# ==============================================
# 셀 10: 전체 통합 테스트용 미니 챗봇
# ==============================================

class MiniDementiaChatbot:
    """테스트용 간단한 챗봇 - 핵심 기능만 포함"""
    
    def __init__(self, config: ChatbotConfig):
        self.config = config
        self.llm = ChatOpenAI(
            model=config.model_name,
            openai_api_key=config.openai_api_key,
            temperature=0.3
        )
        self.vectorizer = TfidfVectorizer(stop_words='english')
    
    def simple_chat(self, message: str) -> Dict:
        """간단한 대화 처리"""
        print(f"\n{'='*50}")
        print(f"사용자 입력: {message}")
        print(f"{'='*50}")
        
        # 1단계: 태스크 유사도 계산
        print("1️⃣ 태스크 유사도 계산...")
        task_scores = {}
        
        for task_name, task_info in ASSESSMENT_TASKS.items():
            prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            평가 영역: {task_name} - {description}
            
            관련성 점수 (0-1): """)
            
            try:
                response = self.llm.invoke(prompt.format_messages(
                    message=message,
                    task_name=task_name,
                    description=task_info["description"]
                ))
                score = float(response.content.strip())
                task_scores[task_name] = score
                print(f"  {task_name}: {score:.2f}")
            except:
                task_scores[task_name] = 0.0
        
        selected_task = max(task_scores.items(), key=lambda x: x[1])
        print(f"🎯 선택된 태스크: {selected_task[0]}")
        
        # 2단계: 맥락 관련성 확인
        print("\n2️⃣ 맥락 관련성 확인...")
        context_prompt = ChatPromptTemplate.from_template("""
        메시지: "{message}"
        태스크: {task_name}
        
        맥락 적합성 (0-1): """)
        
        try:
            response = self.llm.invoke(context_prompt.format_messages(
                message=message,
                task_name=selected_task[0]
            ))
            context_score = float(response.content.strip())
            print(f"📊 맥락 점수: {context_score:.2f}")
        except:
            context_score = 0.0
        
        # 3단계: 응답 생성
        print("\n3️⃣ 응답 생성...")
        
        if context_score >= self.config.context_threshold:
            print("✅ 평가 모드 - 평가 질문 생성")
            
            question_prompt = ChatPromptTemplate.from_template("""
            사용자: "{message}"
            평가 영역: {task_name}
            
            자연스러운 평가 질문 1개만 생성해주세요: """)
            
            try:
                response = self.llm.invoke(question_prompt.format_messages(
                    message=message,
                    task_name=selected_task[0]
                ))
                ai_response = response.content.strip()
                response_type = "assessment"
            except:
                ai_response = "평가 질문 생성에 실패했습니다."
                response_type = "error"
        else:
            print("💬 일상 대화 모드")
            
            casual_prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            
            친근하고 자연스러운 응답을 해주세요: """)
            
            try:
                response = self.llm.invoke(casual_prompt.format_messages(message=message))
                ai_response = response.content.strip()
                response_type = "casual"
            except:
                ai_response = "응답 생성에 실패했습니다."
                response_type = "error"
        
        result = {
            "user_message": message,
            "selected_task": selected_task[0],
            "task_score": selected_task[1],
            "context_score": context_score,
            "response_type": response_type,
            "ai_response": ai_response,
            "threshold": self.config.context_threshold
        }
        
        print(f"\n🤖 AI 응답: {ai_response}")
        print(f"📋 응답 타입: {response_type}")
        
        return result

# 미니 챗봇 테스트
print("=== 미니 챗봇 통합 테스트 ===")

mini_chatbot = MiniDementiaChatbot(config)

# 다양한 테스트 메시지
test_scenarios = [
    "어제 저녁에 뭘 먹었는지 기억이 안나요",  # 기억력 평가 예상
    "오늘이 몇 월인지 헷갈려요",              # 시간 지남력 예상  
    "안녕하세요, 날씨가 좋네요",              # 일상 대화 예상
    "말이 잘 안 나와요"                      # 언어 능력 평가 예상
]

# 각 시나리오 테스트
results = []
for scenario in test_scenarios:
    print(f"\n{'🔬 테스트 시나리오':=^60}")
    result = mini_chatbot.simple_chat(scenario)
    results.append(result)

# 결과 요약
print(f"\n{'📊 테스트 결과 요약':=^60}")
for i, result in enumerate(results, 1):
    print(f"\n{i}. 메시지: '{result['user_message']}'")
    print(f"   선택 태스크: {result['selected_task']} (점수: {result['task_score']:.2f})")
    print(f"   맥락 점수: {result['context_score']:.2f}")
    print(f"   응답 타입: {result['response_type']}")
    print(f"   AI 응답: {result['ai_response'][:50]}...")



패키지 임포트 테스트 시작...
✅ os 임포트 성공
✅ typing 임포트 성공
✅ dataclasses 임포트 성공
✅ langchain_openai 임포트 성공
✅ langchain_core.messages 임포트 성공
✅ langchain_core.prompts 임포트 성공
✅ langgraph.graph 임포트 성공
✅ sklearn 임포트 성공
✅ numpy 임포트 성공
✅ json 임포트 성공

🎉 모든 패키지 임포트 성공!
✅ 데이터 구조 정의 완료
✅ 평가 영역 정의 완료
등록된 평가 영역: ['기억력_평가', '시간_지남력', '언어_능력']
OpenAI API 연결 테스트 중...
❌ OpenAI API 연결 실패: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

해결 방법:
1. API_KEY 변수에 실제 OpenAI API 키를 입력하세요
2. 인터넷 연결을 확인하세요
3. API 키가 유효하고 잔액이 있는지 확인하세요
✅ 챗봇 클래스 기본 구조 정의 완료
✅ 챗봇 초기화 완료 (모델: gpt-4o-mini)
❌ LLM 연결 테스트 실패: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'inv

### test 완성 코드 백업

In [ ]:
# ==============================================
# 셀 12: 완전한 챗봇 클래스 (원본 코드)
# ==============================================

class CompleteDementiaChatbot:
    """완전한 기능을 갖춘 치매 평가 챗봇"""
    
    def __init__(self, config: ChatbotConfig):
        self.config = config
        self.llm = ChatOpenAI(
            model=config.model_name,
            openai_api_key=config.openai_api_key,
            temperature=0.3
        )   
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.graph = self._build_graph()
    
    def _calculate_task_similarity(self, state: ConversationState) -> ConversationState:
        """1. LLM이 각 task와의 부합도 계산"""
        message = state["current_message"]
        
        task_scores = {}
        for task_name, task_info in ASSESSMENT_TASKS.items():
            prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            평가 영역: {task_name} - {description}
            
            사용자 메시지가 이 평가 영역과 얼마나 관련이 있는지 0-1 사이의 점수로 평가해주세요.
            0: 전혀 관련 없음, 1: 매우 관련 있음
            
            점수만 반환해주세요 (예: 0.8):
            """)
            
            response = self.llm.invoke(prompt.format_messages(
                message=message,
                task_name=task_name,
                description=task_info["description"]
            ))
            
            try:
                score = float(response.content.strip())
                task_scores[task_name] = score
            except:
                task_scores[task_name] = 0.0
        
        # 2. 가장 유사도가 높은 task 선택
        selected_task = max(task_scores.items(), key=lambda x: x[1])
        
        state["selected_task"] = selected_task[0]
        return state
    
    def _check_context_relevance(self, state: ConversationState) -> ConversationState:
        """3. 맥락 일치도 확인"""
        message = state["current_message"]
        selected_task = state["selected_task"]
        task_info = ASSESSMENT_TASKS[selected_task]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        평가 영역: {task_name} - {description}
        예시 질문들: {examples}
        
        사용자 메시지가 이 평가 영역의 맥락에 얼마나 적합한지 0-1 사이의 점수로 평가해주세요.
        평가 기준:
        - 메시지가 해당 인지 영역과 관련된 내용인가?
        - 자연스러운 평가로 이어질 수 있는 대화인가?
        
        점수만 반환해주세요 (예: 0.7):
        """)
        
        response = self.llm.invoke(prompt.format_messages(
            message=message,
            task_name=selected_task,
            description=task_info["description"],
            examples=", ".join(task_info["example_questions"][:3])
        ))
        
        try:
            relevance_score = float(response.content.strip())
        except:
            relevance_score = 0.0
            
        state["context_relevance_score"] = relevance_score
        return state
    
    def _route_conversation(self, state: ConversationState) -> Literal["casual_chat", "generate_questions"]:
        """맥락 점수에 따른 대화 모드 결정"""
        if state["context_relevance_score"] < self.config.context_threshold:
            return "casual_chat"
        else:
            return "generate_questions"
    
    def _casual_chat(self, state: ConversationState) -> ConversationState:
        """일상 대화 처리"""
        message = state["current_message"]
        
        prompt = ChatPromptTemplate.from_template("""
        다음 메시지에 대해 자연스럽고 친근한 일상 대화로 응답해주세요.
        가능하면 향후 인지 평가로 자연스럽게 이어질 수 있도록 대화를 유도해보세요.
        
        사용자 메시지: "{message}"
        
        응답:
        """)
        
        response = self.llm.invoke(prompt.format_messages(message=message))
        
        state["messages"].append({
            "role": "assistant",
            "content": response.content,
            "type": "casual"
        })
        state["conversation_mode"] = "casual"
        
        return state
    
    def _generate_assessment_questions(self, state: ConversationState) -> ConversationState:
        """AI 질문 생성 (5개 정도)"""
        message = state["current_message"]
        selected_task = state["selected_task"]
        task_info = ASSESSMENT_TASKS[selected_task]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        평가 영역: {task_name}
        예시 질문들: {examples}
        
        위 사용자 메시지의 맥락을 고려하여, {task_name} 평가를 위한 자연스러운 질문 5개를 생성해주세요.
        예시 질문들을 참고하되, 사용자의 메시지와 자연스럽게 이어지도록 만들어주세요.
        
        각 질문을 새 줄로 구분하여 번호 없이 나열해주세요:
        """)
        
        response = self.llm.invoke(prompt.format_messages(
            message=message,
            task_name=selected_task,
            examples="\n".join(task_info["example_questions"])
        ))
        
        generated_questions = [q.strip() for q in response.content.split('\n') if q.strip()]
        state["generated_questions"] = generated_questions
        
        return state
    
    def _select_best_question(self, state: ConversationState) -> ConversationState:
        """4-6. 각 질문과 예시 문항 간 유사성 비교 및 가장 유사도 높은 질문 선택"""
        generated_questions = state["generated_questions"]
        selected_task = state["selected_task"]
        example_questions = ASSESSMENT_TASKS[selected_task]["example_questions"]
        
        # 모든 질문을 벡터화
        all_questions = generated_questions + example_questions
        if len(all_questions) > 1:
            try:
                tfidf_matrix = self.vectorizer.fit_transform(all_questions)
                
                # 생성된 질문들과 예시 질문들 간의 유사도 계산
                generated_vectors = tfidf_matrix[:len(generated_questions)]
                example_vectors = tfidf_matrix[len(generated_questions):]
                
                similarity_matrix = cosine_similarity(generated_vectors, example_vectors)
                
                # 각 생성된 질문의 최대 유사도 점수 계산
                max_similarities = np.max(similarity_matrix, axis=1)
                
                # 가장 높은 유사도를 가진 질문 선택
                best_question_idx = np.argmax(max_similarities)
                selected_question = generated_questions[best_question_idx]
                
                state["selected_question"] = selected_question
                state["similarity_scores"] = max_similarities.tolist()
                
            except:
                # 벡터화 실패 시 첫 번째 질문 선택
                state["selected_question"] = generated_questions[0] if generated_questions else ""
                state["similarity_scores"] = [0.0]
        else:
            state["selected_question"] = generated_questions[0] if generated_questions else ""
            state["similarity_scores"] = [0.0]
            
        return state
    
    def _validate_context(self, state: ConversationState) -> ConversationState:
        """7. 이전 메시지와 문맥 타당도 확인"""
        current_message = state["current_message"]
        selected_question = state["selected_question"]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자의 이전 메시지: "{current_message}"
        생성된 질문: "{selected_question}"
        
        이 질문이 사용자의 이전 메시지와 자연스럽게 이어지는 대화인지 0-1 사이의 점수로 평가해주세요.
        평가 기준:
        - 대화의 자연스러운 흐름
        - 맥락의 연결성
        - 갑작스럽지 않은 전환
        
        점수만 반환해주세요 (예: 0.8):
        """)
        
        response = self.llm.invoke(prompt.format_messages(
            current_message=current_message,
            selected_question=selected_question
        ))
        
        try:
            validity_score = float(response.content.strip())
        except:
            validity_score = 0.0
            
        state["context_validity_score"] = validity_score
        return state
    
    def _route_question_output(self, state: ConversationState) -> Literal["output_question", "fallback_generate"]:
        """타당도에 따른 질문 출력 여부 결정"""
        if state["context_validity_score"] >= self.config.validity_threshold:
            return "output_question"
        else:
            return "fallback_generate"
    
    def _output_assessment_question(self, state: ConversationState) -> ConversationState:
        """8. AI 메시지로 평가 질문 출력"""
        selected_question = state["selected_question"]
        
        state["messages"].append({
            "role": "assistant", 
            "content": selected_question,
            "type": "assessment",
            "task": state["selected_task"]
        })
        state["conversation_mode"] = "assessment"
        
        return state
    
    def _fallback_question_generation(self, state: ConversationState) -> ConversationState:
        """타당도가 낮을 때 대안 질문 생성"""
        message = state["current_message"]
        
        prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        
        위 메시지에 대해 자연스럽고 친근한 응답을 하면서, 
        인지 능력을 간접적으로 평가할 수 있는 질문을 포함해주세요.
        
        응답:
        """)
        
        response = self.llm.invoke(prompt.format_messages(message=message))
        
        state["messages"].append({
            "role": "assistant",
            "content": response.content,
            "type": "fallback"
        })
        
        return state
    
    def _score_response(self, state: ConversationState) -> ConversationState:
        """9. 평가 질문에 대한 답변 채점"""
        if len(state["messages"]) < 2:
            return state
            
        # 마지막 AI 메시지(질문)와 현재 사용자 답변 가져오기
        last_ai_message = None
        for msg in reversed(state["messages"]):
            if msg["role"] == "assistant" and msg.get("type") == "assessment":
                last_ai_message = msg
                break
        
        if not last_ai_message:
            return state
            
        question = last_ai_message["content"]
        answer = state["current_message"]
        task = last_ai_message.get("task", "")
        
        prompt = ChatPromptTemplate.from_template("""
        평가 영역: {task}
        질문: "{question}"
        사용자 답변: "{answer}"
        
        이 답변을 다음 기준으로 0-10점 척도로 평가해주세요:
        - 정확성: 답변이 사실적으로 정확한가?
        - 적절성: 질문에 적합한 답변인가?
        - 인지능력: 답변이 해당 인지 영역의 정상 기능을 보여주는가?
        
        점수와 간단한 평가 이유를 다음 형식으로 반환해주세요:
        점수: X/10
        평가: [간단한 평가 이유]
        """)
        
        response = self.llm.invoke(prompt.format_messages(
            task=task,
            question=question,
            answer=answer
        ))
        
        # 점수 추출
        try:
            score_line = [line for line in response.content.split('\n') if '점수:' in line][0]
            score = float(score_line.split('/')[0].split(':')[1].strip())
            state["assessment_score"] = score
        except:
            state["assessment_score"] = 0.0
        
        # 채점 결과 저장
        state["messages"].append({
            "role": "system",
            "content": f"채점 결과: {response.content}",
            "type": "scoring"
        })
        
        return state
    
    def _build_graph(self):
        """LangGraph 워크플로우 구성"""
        workflow = StateGraph(ConversationState)
        
        # 노드 추가
        workflow.add_node("calculate_task_similarity", self._calculate_task_similarity)
        workflow.add_node("check_context_relevance", self._check_context_relevance)
        workflow.add_node("casual_chat", self._casual_chat)
        workflow.add_node("generate_questions", self._generate_assessment_questions)
        workflow.add_node("select_best_question", self._select_best_question)
        workflow.add_node("validate_context", self._validate_context)
        workflow.add_node("output_question", self._output_assessment_question)
        workflow.add_node("fallback_generate", self._fallback_question_generation)
        workflow.add_node("score_response", self._score_response)
        
        # 엣지 설정
        workflow.set_entry_point("calculate_task_similarity")
        
        workflow.add_edge("calculate_task_similarity", "check_context_relevance")
        
        workflow.add_conditional_edges(
            "check_context_relevance",
            self._route_conversation,
            {
                "casual_chat": "casual_chat",
                "generate_questions": "generate_questions"
            }
        )
        
        workflow.add_edge("generate_questions", "select_best_question")
        workflow.add_edge("select_best_question", "validate_context")
        
        workflow.add_conditional_edges(
            "validate_context",
            self._route_question_output,
            {
                "output_question": "output_question", 
                "fallback_generate": "fallback_generate"
            }
        )
        
        # 종료점 설정
        workflow.add_edge("casual_chat", END)
        workflow.add_edge("output_question", END)
        workflow.add_edge("fallback_generate", END)
        workflow.add_edge("score_response", END)
        
        return workflow.compile()
    
    def process_message(self, message: str, conversation_history: List[Dict] = None) -> Dict:
        """메시지 처리 메인 함수"""
        if conversation_history is None:
            conversation_history = []
            
        # 이전 메시지가 평가 질문이었다면 채점 먼저 진행
        if (conversation_history and 
            conversation_history[-1].get("type") == "assessment"):
            
            scoring_state = ConversationState(
                messages=conversation_history,
                current_message=message,
                selected_task="",
                context_relevance_score=0.0,
                generated_questions=[],
                selected_question="",
                similarity_scores=[],
                context_validity_score=0.0,
                assessment_score=0.0,
                conversation_mode="assessment"
            )
            
            scoring_result = self._score_response(scoring_state)
            conversation_history = scoring_result["messages"]
        
        # 새로운 응답 생성
        initial_state = ConversationState(
            messages=conversation_history,
            current_message=message,
            selected_task="",
            context_relevance_score=0.0,
            generated_questions=[],
            selected_question="",
            similarity_scores=[],
            context_validity_score=0.0,
            assessment_score=0.0,
            conversation_mode="casual"
        )
        
        final_state = self.graph.invoke(initial_state)
        
        return {
            "messages": final_state["messages"],
            "selected_task": final_state.get("selected_task", ""),
            "context_relevance_score": final_state.get("context_relevance_score", 0.0),
            "similarity_scores": final_state.get("similarity_scores", []),
            "context_validity_score": final_state.get("context_validity_score", 0.0),
            "assessment_score": final_state.get("assessment_score", 0.0),
            "conversation_mode": final_state.get("conversation_mode", "casual")
        }

print("✅ 완전한 챗봇 클래스 정의 완료")

# ==============================================
# 셀 13: 완전한 챗봇 테스트
# ==============================================

def test_complete_chatbot():
    """완전한 챗봇 기능 테스트"""
    print("=== 완전한 챗봇 테스트 ===")
    
    # 챗봇 초기화
    complete_chatbot = CompleteDementiaChatbot(config)
    conversation_history = []
    
    # 테스트 시나리오
    test_conversation = [
        "어제 저녁에 뭘 먹었는지 기억이 안나요",  # 평가 질문 생성 예상
        "김치찌개를 먹었습니다",                   # 평가 답변 (채점 진행)
        "안녕하세요, 오늘 날씨가 좋네요",          # 일상 대화
    ]
    
    for i, message in enumerate(test_conversation, 1):
        print(f"\n{'='*60}")
        print(f"대화 턴 {i}: {message}")
        print(f"{'='*60}")
        
        try:
            result = complete_chatbot.process_message(message, conversation_history)
            
            # 최신 AI 응답 출력
            if result["messages"]:
                latest_message = result["messages"][-1]
                if latest_message["role"] == "assistant":
                    print(f"🤖 AI 응답: {latest_message['content']}")
                    print(f"📋 응답 타입: {latest_message.get('type', 'unknown')}")
            
            # 상세 정보 출력
            print(f"\n📊 분석 결과:")
            print(f"  선택된 태스크: {result['selected_task']}")
            print(f"  맥락 관련성: {result['context_relevance_score']:.2f}")
            print(f"  대화 모드: {result['conversation_mode']}")
            
            if result['assessment_score'] > 0:
                print(f"  평가 점수: {result['assessment_score']}/10")
            
            # 대화 히스토리 업데이트
            conversation_history = result["messages"]
            conversation_history.append({
                "role": "human",
                "content": message
            })
            
        except Exception as e:
            print(f"❌ 오류 발생: {e}")
            import traceback
            traceback.print_exc()

# 완전한 챗봇 테스트 실행
# test_complete_chatbot()

print("✅ 완전한 챗봇 테스트 함수 준비 완료")
print("test_complete_chatbot() 함수를 호출하여 전체 기능을 테스트할 수 있습니다.")
